# B4 — Held-out and full-validation evaluation on FSC-147

## Pre-declared decision rule (agreed before any held-out result is computed)

### 1. Scope and data
- **Primary decision set:** the **1,186 held-out** FSC-147 validation images (val minus dev100), never used in development. Its stratum shares are within ~0.5 pp of the natural distribution.
- **Full validation set (1,286, includes dev100):** a second **complete** report with every metric, for comparability with the literature and the B0 baseline; labelled as including development images.
- **dev100:** reference only, to show each system's development → held-out change.
- **Strata S1–S6:** the same GT-count bins as dev100, for reporting.

### 2. Systems
- **Eligible candidates:** SG_v1.1, GPV_R4_v1, GPV_K_v1, H1b_v1 (`results/B4_candidates.json`).
- **Diagnostic only (never selected):** H2_v1.
- **References:** B0, S.
- All systems run from the same held-out caches, in the same environment, with their frozen functions. No parameter changes.

### 3. Pre-run checks (stop if any fails)
- Every frozen function reproduces its dev100 reference counts 100/100.
- The ported `dense_recovery` prompt generation reproduces the dev100 checkpoint prompts (**Part A**, below). Rule: on frozen inputs the port must be **exact**; with regenerated inputs, **proceed** if ≥ 95% of checkpoint prompts are matched within 1 px **and** GPV_R4_v1 counts regenerated from the new prompts differ from the frozen counts by < 0.5 MAE in total — otherwise stop and debug.
- Held-out cache completeness: 1,186/1,186 for every input.

### 4. Metrics (both reports, every system)
- **Primary:** MAE; **co-primary:** RMSE.
- **Robustness:** trimmed MAE (each system's own 1% largest errors removed); NAE; per-stratum MAE.
- **Secondary:** mean signed error; dev100 → held-out change.
- **Uncertainty:** paired bootstrap — resample the 1,186 images with replacement, same images for both systems, 10,000 resamples, seed 0; the 2.5th–97.5th percentiles of the difference give the 95% CI. Applied to MAE, RMSE, trimmed MAE (re-trimmed per resample) and per-stratum MAE (resampling within stratum).

### 5. Selection procedure (on held-out)
**Stage 1 — accuracy.** Leader = lowest MAE. Tied set = leader + candidates whose MAE difference from the leader has a 95% CI including 0. Within the tied set, drop any candidate whose RMSE is significantly worse (CI excludes 0) than the best RMSE in the set.

**Stage 2 — robustness.** Drop any candidate whose trimmed MAE is significantly worse than the best trimmed MAE in the remaining set. **Stratum balance:** a candidate *fails* a stratum if its MAE there is significantly worse than that stratum's best candidate; drop any candidate that fails **3 or more of the 6 strata**.

**Stage 3 — simplicity.** Among the remaining candidates, choose the easiest to adapt to multi-label: GPV_R4_v1 < GPV_K_v1 < SG_v1.1 < H1b_v1.

The chosen candidate is the **primary architecture** for B5; the next one is the fallback. A **clear winner** is a candidate whose MAE difference from every other candidate has a CI excluding 0; effect sizes with CIs are always reported.

### 6. Reporting commitments
All systems and metrics reported regardless of outcome; H2 reported as a test of routing; no re-runs with changed parameters after held-out results are seen (a bug fix is documented and **all** systems are re-run); the FSC-147 test split is not accessed.

## Parts

| Part | Content | GPU |
|---|---|---|
| **A** | Prompt-port verification on frozen dev100 inputs (exact reproduction of `dense_recovery`) | No |
| B | Environment drift test on dev100 (regenerated D6 features) + held-out prompt/mask caching | Yes |
| C | Evaluation, bootstrap, selection | No |

# Part A — `dense_recovery` prompt-port verification (CPU)

`src/pipeline/prompts.py` ports D6B9-strict (B1 D notebook, Step 12D.9) and the F1 fusion with τ = 0.050 (B1 E notebook, Step 13D.2b). Using the **frozen** dev100 inputs — D6 intermediate SAN cache and A3c masks — it must reproduce the End-to-End checkpoint's `dense_recovery` prompts **exactly** (same count per image; every point matched within 1e-3 px). Any mismatch here is a porting bug, not environment drift.

Expected totals: D6B9-strict 4,551 points; F1 5,744 prompts (5,744 − 4,551 = 1,193 C1 centroids added).

In [1]:
# A.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step A.1 — Paths and modules
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
DEV100_CSV = RESULTS_ROOT / "B1_experiments" / "subsets" / "FSC147_val_dev_100.csv"
D6_CACHE = PROJECT_ROOT / "cache" / "B1_SAN" / "D6_intermediate" / "FSC147_val_dev_100"
B4_DIR = RESULTS_ROOT / "B4_HeldOut_FSC147_Validation"
B4_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.prompts as prompts
for m in (io_cache, prompts):
    importlib.reload(m)

dev100_df = pd.read_csv(DEV100_CSV).sort_values("dataset_index").reset_index(drop=True)
assert D6_CACHE.exists(), D6_CACHE
print("Repo commit:", REPO_COMMIT)

Repo commit: 4c852ac24702dcceda1bfb1839ebaf8dc197dfb2


In [3]:
# ============================================================
# Step A.2 — Regenerate dense_recovery from frozen inputs; compare
# ============================================================

def match_points(a, b):
    # Optimal one-to-one matching; returns distances of matched pairs
    a = np.asarray(a, float).reshape(-1, 2); b = np.asarray(b, float).reshape(-1, 2)
    if len(a) == 0 or len(b) == 0:
        return np.zeros(0)
    d = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
    r, c = linear_sum_assignment(d)
    return d[r, c]

rows = []
t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Port check"):
    di = row.dataset_index
    d6 = torch.load(D6_CACHE / f"{io_cache.image_stem(row.image_id)}.pt",
                    map_location="cpu", weights_only=False)
    a3c, _ = io_cache.load_a3c_mask(io_cache.cache_paths(di, row.image_id)["a3c_masks"])
    pts, info = prompts.dense_recovery_points(a3c, d6["mask_preds"], d6["target_query_prob"])

    ckpt = io_cache.load_guided_masks(io_cache.guided_cache_path("dense_recovery", di, row.image_id))
    ref = ckpt["prompts_xy"]
    dist = match_points(pts, ref)
    rows.append({"dataset_index": di, "image_id": row.image_id,
                 "density_stratum": row.density_stratum,
                 "n_ported": len(pts), "n_checkpoint": len(ref),
                 "n_d6b9": info["n_d"], "n_c_added": info["n_c_added"],
                 "matched_1e3": int((dist <= 1e-3).sum()), "matched_1px": int((dist <= 1.0).sum()),
                 "max_match_dist": float(dist.max()) if len(dist) else 0.0})
port_df = pd.DataFrame(rows)
print(f"Runtime: {time.time() - t0:.0f} s")

tot = port_df[["n_ported", "n_checkpoint", "n_d6b9", "n_c_added", "matched_1e3", "matched_1px"]].sum()
count_ok = int((port_df["n_ported"] == port_df["n_checkpoint"]).sum())
exact_imgs = int(((port_df["n_ported"] == port_df["n_checkpoint"])
                  & (port_df["matched_1e3"] == port_df["n_checkpoint"])).sum())
print(f"D6B9-strict points : {int(tot['n_d6b9'])} (expected 4551)")
print(f"C1 centroids added : {int(tot['n_c_added'])} (expected 1193)")
print(f"Prompts ported     : {int(tot['n_ported'])} | checkpoint {int(tot['n_checkpoint'])} (expected 5744)")
print(f"Per-image count match           : {count_ok}/100")
print(f"Prompts matched within 1e-3 px  : {int(tot['matched_1e3'])}/{int(tot['n_checkpoint'])}")
print(f"Prompts matched within 1 px     : {int(tot['matched_1px'])}/{int(tot['n_checkpoint'])}")
print(f"Images reproduced exactly       : {exact_imgs}/100")

PORT_EXACT = exact_imgs == 100
print("\nPART A VERDICT:", "PASS — port reproduces dense_recovery exactly" if PORT_EXACT
      else "FAIL — porting difference; inspect the table below before any GPU work")
if not PORT_EXACT:
    display(port_df[(port_df["n_ported"] != port_df["n_checkpoint"])
                    | (port_df["matched_1e3"] != port_df["n_checkpoint"])])

Port check:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 326 s
D6B9-strict points : 4551 (expected 4551)
C1 centroids added : 1193 (expected 1193)
Prompts ported     : 5744 | checkpoint 5744 (expected 5744)
Per-image count match           : 100/100
Prompts matched within 1e-3 px  : 5744/5744
Prompts matched within 1 px     : 5744/5744
Images reproduced exactly       : 100/100

PART A VERDICT: PASS — port reproduces dense_recovery exactly


In [4]:
# ============================================================
# Step A.3 — Save
# ============================================================

A_DIR = B4_DIR / "partA_prompt_port"
A_DIR.mkdir(parents=True, exist_ok=True)
port_df.to_csv(A_DIR / "B4_A_port_check_per_image.csv", index=False)
with open(A_DIR / "B4_A_metadata.json", "w") as f:
    json.dump({"part": "A — dense_recovery prompt-port verification (frozen dev100 inputs)",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "module": "src/pipeline/prompts.py",
               "expected": {"d6b9_points": 4551, "c_added": 1193, "prompts": 5744},
               "observed": {k: int(v) for k, v in tot.items()},
               "images_exact": exact_imgs, "port_exact": PORT_EXACT,
               "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", A_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B4_HeldOut_FSC147_Validation/partA_prompt_port


## Part A result

**PASS — exact.** On frozen dev100 inputs the port reproduces `dense_recovery` exactly: D6B9-strict 4,551/4,551, C1 centroids added 1,193/1,193, prompts 5,744/5,744, all matched within 1e-3 px, 100/100 images. No porting bugs; any later difference can only come from regenerating inputs.

# Part B — Environment drift test (dev100) and held-out guided-mask caching (GPU)

## Steps

| Step | Content |
|---|---|
| B.1 | Install SAN + detectron2 (protobuf 3.20.3, iopath 0.1.10 pins) and SAM2 (frozen commit) — as B3R |
| B.2 | Build SAN and the SAM2 predictor (as the End-to-End checkpoint: default `build_sam2`, `SAM2ImagePredictor`, image = `np.array(PIL RGB)`) |
| B.3 | **D6 regeneration check (dev100):** regenerated D6 intermediates vs the frozen D6 cache |
| B.4 | **Drift test (dev100), pre-declared rule:** regenerate D6 + SAN maps → A3c → `dense_recovery` prompts → SAM2 masks → GPV_R4_v1 counts (frozen B0 masks). **Proceed** if ≥ 95% of checkpoint prompts are matched within 1 px **and** Σ|GPV_R4 count difference| / 100 < 0.5. Otherwise stop. |
| B.5 | **Held-out caching (resumable):** for the 1,186 held-out images — D6 intermediates, `dense_recovery` prompts (A3c from the held-out cache), SAM2 guided masks — saved in the checkpoint format |
| B.6 | Validate the held-out guided cache and write an index |

No GT is loaded for held-out images; no counting or evaluation happens in Part B.

## Run order

**T4 runtime** → Part A cells A.0–A.1 (setup) → B.1 → (restart if B.2 asks) → B.2 … B.6. Expected: installs ~15 min, B.3–B.4 ~10 min, B.5 ~45–75 min.

In [ ]:
# ============================================================
# Step B.1 — Install SAN + detectron2 and SAM2 (as B3R)
# ============================================================

import importlib.metadata as md_meta
import src.pipeline.b0 as b0
importlib.reload(b0)
assert torch.cuda.is_available(), "Switch the runtime to T4."

DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
FROZEN_SAM2_COMMIT = b0.FROZEN_SAM2_COMMIT

%cd /content
!rm -rf /content/SAN /content/sam2_repo
!git clone -q https://github.com/MendelXu/SAN.git /content/SAN
!pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0"
!pip uninstall -y -q detectron2
!python -m pip install -q "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"

!git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
!git -C /content/sam2_repo checkout -q {FROZEN_SAM2_COMMIT}
%cd /content/sam2_repo
!pip install -q -e .
%cd /content

# Restore the ABC package versions (see B3R erratum on protobuf / iopath)
!pip install -q "protobuf==3.20.3" "iopath==0.1.10"

IMPORT_TEST = r'''
import sys
sys.path.insert(0, "/content/SAN"); sys.path.insert(0, "/content/sam2_repo")
import google.protobuf, wandb, open_clip, detectron2, san
from sam2.build_sam import build_sam2
print("ALL IMPORTS OK", google.protobuf.__version__)
'''
r = subprocess.run([sys.executable, "-c", IMPORT_TEST], cwd="/content", capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-2000:])
assert r.returncode == 0, "Import test failed"
print("If B.2 reports a protobuf mismatch: Runtime -> Restart session, re-run A.0-A.1, then B.2.")

/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver doe

In [ ]:
# ============================================================
# Step B.2 — Build SAN and the SAM2 predictor
# ============================================================

import shutil
import google.protobuf
assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
    "protobuf mismatch — restart the session, re-run A.0-A.1, then B.2."

for p in ["/content/sam2_repo", "/content/SAN"]:
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

import src.pipeline.semantic as semantic
import src.pipeline.gpv as gpv
for m in (semantic, gpv, io_cache, prompts):
    importlib.reload(m)
assert hasattr(semantic, "run_san_d6") and hasattr(io_cache, "save_guided_masks"), \
    "Old modules — commit the Part B versions of semantic.py and io_cache.py."

DEVICE = "cuda"
SAN_ROOT = Path("/content/SAN")
SAN_CKPT_DRIVE = PROJECT_ROOT / "Checkpoints" / "SAN" / "san_vit_b_16.pth"
SAN_CKPT_LOCAL = SAN_ROOT / "output" / "san_vit_b_16.pth"
SAN_CKPT_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if not SAN_CKPT_LOCAL.exists():
    shutil.copy2(SAN_CKPT_DRIVE, SAN_CKPT_LOCAL)
san_model = semantic.build_san(SAN_ROOT, SAN_CKPT_LOCAL, device=DEVICE)

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
SAM2_CHECKPOINT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"
assert b0.sha256_file(SAM2_CHECKPOINT) == b0.FROZEN_CHECKPOINT_SHA256
sam2_model = build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_CHECKPOINT), device=DEVICE)
sam2_model.eval()
predictor = SAM2ImagePredictor(sam2_model)

ENV_B = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda,
         "san_commit": subprocess.check_output(["git", "-C", "/content/SAN", "rev-parse", "HEAD"], text=True).strip(),
         "sam2_commit": subprocess.check_output(["git", "-C", "/content/sam2_repo", "rev-parse", "HEAD"], text=True).strip(),
         **{k: md_meta.version(k) for k in ["protobuf", "iopath", "detectron2", "open_clip_torch"]}}
print(ENV_B)


def guided_masks_for_prompts(image_np, prompts_xy):
    # End-to-End checkpoint protocol: one positive point -> one mask (multimask False)
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|████████████████████████████████████████| 351M/351M [00:01<00:00, 214MiB/s]
/content/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


{'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'san_commit': 'e19262d1a23992fc2cef934d6756b89ff50fe5c3', 'sam2_commit': '2b90b9f5ceec907a1c18123530e92e794ad901a4', 'protobuf': '3.20.3', 'iopath': '0.1.10', 'detectron2': '0.6', 'open_clip_torch': '2.16.0'}


In [ ]:
# ============================================================
# Step B.3 — D6 regeneration check (dev100)
# ============================================================

from PIL import Image
FSC147_IMAGE_DIR = PROJECT_ROOT / "datasets" / "FSC147" / "raw" / "images_384_VarV2"

d6_rows, FRESH = [], {}
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="D6 regen"):
    img = Image.open(FSC147_IMAGE_DIR / row.image_id).convert("RGB")
    mp, tq = semantic.run_san_d6(san_model, img, row.category, DEVICE)
    target, _ = semantic.run_san(san_model, img, row.category, DEVICE)
    a3c, _ = semantic.a3c_mask(target)
    frozen = torch.load(D6_CACHE / f"{io_cache.image_stem(row.image_id)}.pt",
                        map_location="cpu", weights_only=False)
    FRESH[row.dataset_index] = {"mask_preds": mp, "tqp": tq, "a3c": a3c}
    d6_rows.append({"dataset_index": row.dataset_index,
                    "shape_ok": tuple(mp.shape) == tuple(frozen["mask_preds"].shape),
                    "mask_preds_max_absdiff": float((mp - frozen["mask_preds"].float()).abs().max()),
                    "tqp_max_absdiff": float((tq - frozen["target_query_prob"].float()).abs().max()),
                    "exact": bool(torch.equal(mp, frozen["mask_preds"].float())
                                  and torch.equal(tq, frozen["target_query_prob"].float()))})
d6_df = pd.DataFrame(d6_rows)
print(f"Shapes match           : {int(d6_df['shape_ok'].sum())}/100")
print(f"Bit-identical images   : {int(d6_df['exact'].sum())}/100")
print(f"mask_preds max |diff|  : {d6_df['mask_preds_max_absdiff'].max():.3e}")
print(f"P(t|q) max |diff|      : {d6_df['tqp_max_absdiff'].max():.3e}")

D6 regen:   0%|          | 0/100 [00:00<?, ?it/s]

Shapes match           : 100/100
Bit-identical images   : 58/100
mask_preds max |diff|  : 7.248e-05
P(t|q) max |diff|      : 2.682e-06


In [ ]:
# ============================================================
# Step B.4 — Drift test (dev100): prompts -> SAM2 -> GPV_R4_v1 counts
# ============================================================

if "match_points" not in globals():
    def match_points(a, b):
        a = np.asarray(a, float).reshape(-1, 2); b = np.asarray(b, float).reshape(-1, 2)
        if len(a) == 0 or len(b) == 0:
            return np.zeros(0)
        d = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
        r, c = linear_sum_assignment(d)
        return d[r, c]

GPV_FROZEN = pd.read_csv(RESULTS_ROOT / "B3GK_GPV_Multiplicity_and_Prompts" / "frozen"
                         / "GPV_frozen_dev100_reference_counts.csv").set_index("dataset_index")

drift_rows = []
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Drift test"):
    di = row.dataset_index
    f = FRESH[di]
    pts, info = prompts.dense_recovery_points(f["a3c"], f["mask_preds"], f["tqp"])
    ref = io_cache.load_guided_masks(io_cache.guided_cache_path("dense_recovery", di, row.image_id))["prompts_xy"]
    dist = match_points(pts, ref)

    image_np = np.array(Image.open(FSC147_IMAGE_DIR / row.image_id).convert("RGB"), copy=True)
    gm, gs = guided_masks_for_prompts(image_np, pts)
    keep = gpv.nms_masks(gm, gs, 0.5)
    b0_masks, b0_meta = io_cache.load_b0_masks(io_cache.cache_paths(di, row.image_id)["b0_masks"])
    r4_m, _, _ = gpv.build_r4(gm[keep], gs[keep], b0_masks, b0_meta["predicted_iou"], 0.5)
    drift_rows.append({"dataset_index": di, "density_stratum": row.density_stratum,
                       "n_fresh": len(pts), "n_ckpt": len(ref),
                       "matched_1px": int((dist <= 1.0).sum()),
                       "R4_fresh": int(r4_m.shape[0]),
                       "R4_frozen": int(GPV_FROZEN.loc[di, "GPV_R4_v1_count"])})
drift_df = pd.DataFrame(drift_rows)

share_1px = drift_df["matched_1px"].sum() / drift_df["n_ckpt"].sum()
count_mae_diff = float((drift_df["R4_fresh"] - drift_df["R4_frozen"]).abs().mean())
print(f"Prompts fresh / checkpoint      : {int(drift_df['n_fresh'].sum())} / {int(drift_df['n_ckpt'].sum())}")
print(f"Checkpoint prompts matched 1 px : {share_1px:.1%}  (rule: >= 95%)")
print(f"GPV_R4 count |diff| per image   : {count_mae_diff:.3f}  (rule: < 0.5)")
print(f"Images with identical R4 count  : {int((drift_df['R4_fresh'] == drift_df['R4_frozen']).sum())}/100")
DRIFT_PASS = share_1px >= 0.95 and count_mae_diff < 0.5
print("\nDRIFT TEST:", "PASS — proceed to held-out caching" if DRIFT_PASS
      else "FAIL — stop and investigate before B.5")
if not DRIFT_PASS:
    display(drift_df[drift_df["R4_fresh"] != drift_df["R4_frozen"]])

Drift test:   0%|          | 0/100 [00:00<?, ?it/s]

Prompts fresh / checkpoint      : 5744 / 5744
Checkpoint prompts matched 1 px : 100.0%  (rule: >= 95%)
GPV_R4 count |diff| per image   : 0.000  (rule: < 0.5)
Images with identical R4 count  : 100/100

DRIFT TEST: PASS — proceed to held-out caching


In [ ]:
# ============================================================
# Step B.5 — Held-out D6 + dense_recovery prompts + SAM2 guided masks (resumable)
# ============================================================

assert DRIFT_PASS, "Drift test failed — do not cache held-out."
HO_ROOT = PROJECT_ROOT / "cache" / "FSC147_val_heldout_1186"
HO_DIRS = {"b0_masks": HO_ROOT / "b0_masks", "a3c_masks": HO_ROOT / "a3c_masks",
           "depth": HO_ROOT / "depth", "m1_split_labels": HO_ROOT / "m1_split_labels"}
HO_D6 = HO_ROOT / "d6_intermediate"
HO_GUIDED = HO_ROOT / "guided_dense_recovery"
HO_DONE = HO_ROOT / "_done_guided"
for d in (HO_D6, HO_GUIDED, HO_DONE):
    d.mkdir(parents=True, exist_ok=True)
heldout_df = pd.read_csv(HO_ROOT / "FSC147_val_heldout_1186.csv")
assert len(heldout_df) == 1186


def guided_path(di, image_id):
    return HO_GUIDED / f"{int(di):04d}_{io_cache.image_stem(image_id)}.npz"

def done_path(di, image_id):
    return HO_DONE / f"{int(di):05d}_{io_cache.image_stem(image_id)}.json"


todo = [r for r in heldout_df.itertuples(index=False) if not done_path(r.dataset_index, r.image_id).exists()]
print(f"Already done: {len(heldout_df) - len(todo)} | to process: {len(todo)}")
errors, t0 = [], time.time()
for i, r in enumerate(tqdm(todo, desc="Held-out guided")):
    try:
        img = Image.open(FSC147_IMAGE_DIR / r.image_id).convert("RGB")
        mp, tq = semantic.run_san_d6(san_model, img, r.category, DEVICE)
        torch.save({"image_id": r.image_id, "target_category": r.category,
                    "mask_preds": mp, "target_query_prob": tq},
                   HO_D6 / f"{io_cache.image_stem(r.image_id)}.pt")
        a3c, _ = io_cache.load_a3c_mask(io_cache.cache_paths(r.dataset_index, r.image_id, HO_DIRS)["a3c_masks"])
        pts, info = prompts.dense_recovery_points(a3c, mp, tq)
        image_np = np.array(img, copy=True)
        gm, gs = guided_masks_for_prompts(image_np, pts)
        io_cache.save_guided_masks(guided_path(r.dataset_index, r.image_id), gm, pts, gs,
                                   image_np.shape[:2], dataset_index=r.dataset_index,
                                   image_id=r.image_id)
        with open(done_path(r.dataset_index, r.image_id), "w") as f:
            json.dump({"dataset_index": int(r.dataset_index), "image_id": r.image_id,
                       "n_prompts": int(len(pts)), **info}, f)
    except Exception as e:
        errors.append({"image_id": r.image_id, "error": repr(e)})
        print("ERROR", r.image_id, repr(e))
    if (i + 1) % 50 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed {(time.time() - t0) / 60:.1f} min | errors {len(errors)}")

Already done: 0 | to process: 1186


Held-out guided:   0%|          | 0/1186 [00:00<?, ?it/s]

Elapsed 51.9 min | errors 0


In [ ]:
# ============================================================
# Step B.6 — Validate held-out guided cache and save Part B record
# ============================================================

idx_rows, problems = [], []
for r in tqdm(heldout_df.itertuples(index=False), total=len(heldout_df), desc="Validate"):
    dp = done_path(r.dataset_index, r.image_id)
    if not dp.exists():
        problems.append((r.image_id, "not done")); continue
    try:
        g = io_cache.load_guided_masks(guided_path(r.dataset_index, r.image_id))
        meta = json.load(open(dp))
        assert g["masks"].shape[0] == len(g["prompts_xy"]) == meta["n_prompts"]
        idx_rows.append({"dataset_index": r.dataset_index, "image_id": r.image_id,
                         "n_prompts": meta["n_prompts"], "n_d6b9": meta["n_d"],
                         "n_c_added": meta["n_c_added"]})
    except Exception as e:
        problems.append((r.image_id, repr(e)))
ho_guided_index = pd.DataFrame(idx_rows)
ho_guided_index.to_csv(HO_ROOT / "guided_dense_recovery_index.csv", index=False)

B_DIR = B4_DIR / "partB_drift_and_caching"
B_DIR.mkdir(parents=True, exist_ok=True)
d6_df.to_csv(B_DIR / "B4_B_d6_regeneration_dev100.csv", index=False)
drift_df.to_csv(B_DIR / "B4_B_drift_test_dev100.csv", index=False)
with open(B_DIR / "B4_B_metadata.json", "w") as f:
    json.dump({"part": "B — drift test and held-out guided caching",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "environment": ENV_B,
               "d6_bit_identical_images": int(d6_df["exact"].sum()),
               "drift": {"share_prompts_1px": float(share_1px),
                         "gpv_r4_count_absdiff_per_image": count_mae_diff, "pass": bool(DRIFT_PASS)},
               "heldout_guided_valid": len(ho_guided_index), "problems": len(problems),
               "gt_loaded_for_heldout": False, "test_set_accessed": False}, f, indent=2)

complete = len(ho_guided_index) == 1186 and not problems
print(f"Held-out guided cache: {len(ho_guided_index)}/1186 valid | problems: {len(problems)}")
print(f"Total held-out prompts: {int(ho_guided_index['n_prompts'].sum()) if len(ho_guided_index) else 0}")
print("PART B VERDICT:", "COMPLETE — ready for Part C" if complete
      else "INCOMPLETE — re-run B.5 (resumes), then B.6")

Validate:   0%|          | 0/1186 [00:00<?, ?it/s]

Held-out guided cache: 1186/1186 valid | problems: 0
Total held-out prompts: 68747
PART B VERDICT: COMPLETE — ready for Part C


## Part B result

- **D6 regeneration:** shapes 100/100; 58/100 bit-identical; max |diff| 7.2e-5 (mask logits), 2.7e-6 (P(t|q)) — CUDA 13.0 floating-point drift.
- **Drift test: PASS** — 100% of checkpoint prompts matched within 1 px; GPV_R4 counts identical on 100/100 dev100 images.
- **Held-out guided cache:** 1,186/1,186 valid, 0 errors, 68,747 prompts (≈ 58 per image, dev100 57.4); 52 min.

# Part C — Evaluation and selection (CPU)

| Step | Content |
|---|---|
| C.1 | Setup; GT counts; strata (rank-based qcut on full-val GT, as in the ABC notebook) — must reproduce dev100's labels 100/100 |
| C.2 | **Pre-run check:** every frozen function reproduces its dev100 reference counts (frozen caches) |
| C.3 | dev100 in the **current environment** (B3R fresh caches) — for the full-validation report |
| C.4 | **Held-out:** all systems on 1,186 images (resumable) |
| C.5 | Metrics — held-out and full-val reports |
| C.6 | Paired bootstrap (10,000, seed 0; within-stratum resampling seeds 1000–1005) |
| C.7 | **Pre-declared selection procedure** (held-out) — primary + fallback |
| C.8 | Save |

GT is loaded only here, for evaluation. Systems: SG_v1.1, GPV_R4_v1, GPV_K_v1, H1b_v1 (eligible); H2_v1 (diagnostic); B0, S (references).

**Run order:** CPU runtime → A.0–A.1 → Part C. (Parts A–B not re-run.)

In [5]:
# ============================================================
# Step C.1 — Setup, GT counts and strata
# ============================================================

import importlib
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.multiplicity as K
import src.pipeline.hybrid as hybrid
import src.datasets as datasets_mod
for m in (io_cache, sg, gpv, K, hybrid, datasets_mod):
    importlib.reload(m)

FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
HO_ROOT = PROJECT_ROOT / "cache" / "FSC147_val_heldout_1186"
HO_DIRS = {"b0_masks": HO_ROOT / "b0_masks", "a3c_masks": HO_ROOT / "a3c_masks",
           "depth": HO_ROOT / "depth", "m1_split_labels": HO_ROOT / "m1_split_labels"}
FRESH_ROOT = RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "section3_end_to_end" / "fresh_dev100_cache"
FRESH_DIRS = {"b0_masks": FRESH_ROOT / "b0_masks", "a3c_masks": FRESH_ROOT / "a3c_masks",
              "depth": FRESH_ROOT / "depth", "m1_split_labels": FRESH_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")
assert SG_CFG.variant == "v1.1"

C_DIR = B4_DIR / "partC_evaluation"
C_DIR.mkdir(parents=True, exist_ok=True)

# ---- GT counts for the whole val split (evaluation only)
val_ds = datasets_mod.FSC147Dataset(FSC147_RAW, split="val", load_images=False)
val_meta = pd.DataFrame([{"dataset_index": i, "image_id": val_ds.image_ids[i],
                          "category": val_ds.image_to_class[val_ds.image_ids[i]],
                          "gt_count": len(val_ds.annotations[val_ds.image_ids[i]]["points"])}
                         for i in range(len(val_ds))])

# ---- Strata exactly as the ABC notebook: rank(method="first") + qcut on full-val GT
ranked = val_meta["gt_count"].rank(method="first")
STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
val_meta["density_stratum"] = pd.qcut(ranked, q=[0, .2, .4, .6, .8, .95, 1.0], labels=STRATA).astype(str)

chk = dev100_df.merge(val_meta[["dataset_index", "density_stratum", "gt_count"]],
                      on="dataset_index", suffixes=("", "_c"))
strata_ok = int((chk["density_stratum"] == chk["density_stratum_c"]).sum())
gt_ok = int((chk["gt_count"] == chk["gt_count_c"]).sum())
print(f"dev100 strata reproduced : {strata_ok}/100  {'PASS' if strata_ok == 100 else 'FAIL'}")
print(f"dev100 GT reproduced     : {gt_ok}/100")

heldout_ids = set(pd.read_csv(HO_ROOT / "FSC147_val_heldout_1186.csv")["dataset_index"])
val_meta["subset"] = np.where(val_meta["dataset_index"].isin(heldout_ids), "heldout", "dev100")
print(val_meta.groupby(["subset", "density_stratum"]).size().unstack(0).reindex(STRATA))

dev100 strata reproduced : 100/100  PASS
dev100 GT reproduced     : 100/100
subset           dev100  heldout
density_stratum                 
S1_0_20              20      238
S2_20_40             20      237
S3_40_60             20      237
S4_60_80             20      237
S5_80_95             10      182
S6_95_100            10       55


In [6]:
# ============================================================
# Step C.1b — One function computing every system for one image
# ============================================================

def compute_systems(b0_masks, b0_iou, guided, a3c, depth, san_target):
    out = {"B0": int(b0_masks.shape[0])}
    s, _ = sg.compute_sg_image(b0_masks, a3c, depth, SG_CFG)
    out.update({"S": int(s["S_count"]), "SG_v1.1": float(s["SG_count"]), "sg_gate": bool(s["stage2_on"])})
    keep = gpv.nms_masks(guided["masks"], guided["scores"], 0.5)
    r4_m, r4_s, _ = gpv.build_r4(guided["masks"][keep], guided["scores"][keep], b0_masks, b0_iou, 0.5)
    out["GPV_R4_v1"] = int(r4_m.shape[0])
    d1 = gpv.d1_peaks(san_target)
    k_count, gate1 = K.gpv_k_count(r4_m, r4_s, d1, depth)
    out.update({"GPV_K_v1": float(k_count), "gate1": bool(gate1)})
    h1b, _ = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    out["H1b_v1"] = float(h1b)
    out["H2_v1"] = hybrid.h2_route(out["SG_v1.1"], out["sg_gate"], out["GPV_K_v1"], out["gate1"])
    return out


def load_inputs(di, image_id, dirs, san_path, guided_path):
    p = io_cache.cache_paths(di, image_id, dirs)
    b0_masks, b0_meta = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    depth = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(san_path)
    guided = io_cache.load_guided_masks(guided_path)
    return b0_masks, b0_meta["predicted_iou"], guided, a3c, depth, san["target_score"]

SYSTEMS_ALL = ["B0", "S", "SG_v1.1", "GPV_R4_v1", "GPV_K_v1", "H1b_v1", "H2_v1"]
ELIGIBLE = ["SG_v1.1", "GPV_R4_v1", "GPV_K_v1", "H1b_v1"]
SIMPLICITY = ["GPV_R4_v1", "GPV_K_v1", "SG_v1.1", "H1b_v1"]
print("compute_systems ready")

compute_systems ready


In [7]:
# ============================================================
# Step C.2 — Pre-run check: frozen functions reproduce dev100 references
# ============================================================

ref_sg = pd.read_csv(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "section4_SG_v1_1"
                     / "B3R_s4_SG_v1_1_per_image.csv").set_index("dataset_index")["SG_count"]
ref_gpv = pd.read_csv(RESULTS_ROOT / "B3GK_GPV_Multiplicity_and_Prompts" / "frozen"
                      / "GPV_frozen_dev100_reference_counts.csv").set_index("dataset_index")
ref_h = pd.read_csv(RESULTS_ROOT / "B3H_Hybrid_SG_GPV" / "frozen"
                    / "H_frozen_dev100_reference_counts.csv").set_index("dataset_index")

rows, t0 = [], time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Frozen dev100"):
    di = row.dataset_index
    inp = load_inputs(di, row.image_id, None,
                      io_cache.SAN_DEV100_CACHE_DIR / io_cache.san_scores_filename(row.image_id),
                      io_cache.guided_cache_path("dense_recovery", di, row.image_id))
    rows.append({"dataset_index": di, **compute_systems(*inp)})
frozen_check = pd.DataFrame(rows).set_index("dataset_index")

checks = {
    "SG_v1.1": np.isclose(frozen_check["SG_v1.1"], ref_sg.loc[frozen_check.index]),
    "GPV_R4_v1": frozen_check["GPV_R4_v1"] == ref_gpv.loc[frozen_check.index, "GPV_R4_v1_count"],
    "GPV_K_v1": np.isclose(frozen_check["GPV_K_v1"], ref_gpv.loc[frozen_check.index, "GPV_K_v1_count"]),
    "H1b_v1": np.isclose(frozen_check["H1b_v1"], ref_h.loc[frozen_check.index, "H1b_v1_count"]),
    "H2_v1": np.isclose(frozen_check["H2_v1"], ref_h.loc[frozen_check.index, "H2_v1_count"]),
}
print(f"Runtime: {time.time() - t0:.0f} s")
for k, v in checks.items():
    print(f"  {k:<10}: {int(np.sum(v))}/100  {'PASS' if np.all(v) else 'FAIL'}")
PRECHECK_PASS = all(np.all(v) for v in checks.values())
print("PRE-RUN CHECK:", "PASS" if PRECHECK_PASS else "FAIL — stop")

Frozen dev100:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 425 s
  SG_v1.1   : 100/100  PASS
  GPV_R4_v1 : 100/100  PASS
  GPV_K_v1  : 100/100  PASS
  H1b_v1    : 100/100  PASS
  H2_v1     : 100/100  PASS
PRE-RUN CHECK: PASS


In [8]:
# ============================================================
# Step C.3 — dev100 in the current environment (for the full-val report)
# ============================================================

assert PRECHECK_PASS
rows = []
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="dev100 current env"):
    di = row.dataset_index
    inp = load_inputs(di, row.image_id, FRESH_DIRS,
                      FRESH_ROOT / "san" / io_cache.san_scores_filename(row.image_id),
                      io_cache.guided_cache_path("dense_recovery", di, row.image_id))
    rows.append({"dataset_index": di, **compute_systems(*inp)})
dev_current = pd.DataFrame(rows)
diffs = {s: int((~np.isclose(dev_current.set_index("dataset_index")[s],
                             frozen_check[s].loc[dev_current["dataset_index"]])).sum())
         for s in SYSTEMS_ALL}
print("dev100 images whose count differs, current vs frozen environment:", diffs)

dev100 current env:   0%|          | 0/100 [00:00<?, ?it/s]

dev100 images whose count differs, current vs frozen environment: {'B0': 0, 'S': 0, 'SG_v1.1': 4, 'GPV_R4_v1': 0, 'GPV_K_v1': 7, 'H1b_v1': 4, 'H2_v1': 10}


In [9]:
# ============================================================
# Step C.4 — Held-out: all systems on 1,186 images (resumable)
# ============================================================

HO_RES = C_DIR / "heldout_rows"
HO_RES.mkdir(parents=True, exist_ok=True)
ho_list = pd.read_csv(HO_ROOT / "FSC147_val_heldout_1186.csv")
todo = [r for r in ho_list.itertuples(index=False)
        if not (HO_RES / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(ho_list) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for r in tqdm(todo, desc="Held-out systems"):
    di = r.dataset_index
    inp = load_inputs(di, r.image_id, HO_DIRS,
                      HO_ROOT / "san" / io_cache.san_scores_filename(r.image_id),
                      HO_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{io_cache.image_stem(r.image_id)}.npz")
    res = compute_systems(*inp)
    with open(HO_RES / f"{int(di):05d}.json", "w") as f:
        json.dump({"dataset_index": int(di), "image_id": r.image_id, **res}, f)
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

ho_res = pd.DataFrame([json.load(open(p)) for p in sorted(HO_RES.glob("*.json"))])
assert len(ho_res) == 1186, len(ho_res)

Done: 0 | to process: 1186


Held-out systems:   0%|          | 0/1186 [00:00<?, ?it/s]

Elapsed: 106.7 min


In [10]:
# ============================================================
# Step C.5 — Metrics: held-out and full-val reports
# ============================================================

ho = ho_res.merge(val_meta, on=["dataset_index", "image_id"], validate="one_to_one")
dv = dev_current.merge(val_meta, on="dataset_index", validate="one_to_one")
full = pd.concat([ho, dv], ignore_index=True)
assert len(ho) == 1186 and len(full) == 1286

def report(df, label):
    gt = df["gt_count"].to_numpy(float)
    st = df["density_stratum"].to_numpy()
    k = max(1, int(round(0.01 * len(df))))
    rows = []
    for s in SYSTEMS_ALL:
        err = df[s].to_numpy(float) - gt
        ae = np.abs(err)
        row = {"set": label, "system": s, "role": ("eligible" if s in ELIGIBLE else
               "diagnostic" if s == "H2_v1" else "reference"),
               "MAE": ae.mean(), "RMSE": np.sqrt((err ** 2).mean()),
               "trimmed_MAE": np.sort(ae)[:len(ae) - k].mean(),
               "NAE": (ae / gt).mean(), "mean_signed_error": err.mean()}
        for sname in STRATA:
            row[f"MAE_{sname[:2]}"] = ae[st == sname].mean()
        rows.append(row)
    return pd.DataFrame(rows)

rep_ho = report(ho, "held-out (1,186)")
rep_full = report(full, "full val (1,286)")
COLS_R = ["system", "role", "MAE", "RMSE", "trimmed_MAE", "NAE", "mean_signed_error"] + [f"MAE_{s[:2]}" for s in STRATA]
print("HELD-OUT (primary decision set)")
display(rep_ho[COLS_R].round(2))
print("\nFULL VALIDATION (includes dev100 development images)")
display(rep_full[COLS_R].round(2))

dev_ref = report(dev_current.merge(val_meta, on="dataset_index"), "dev100 (current env)")
drop = rep_ho.set_index("system")["MAE"] - dev_ref.set_index("system")["MAE"]
print("\nHeld-out MAE minus dev100 MAE (negative = lower error on held-out):")
display(drop.round(2).to_frame("held-out − dev100"))

HELD-OUT (primary decision set)


,system,role,MAE,RMSE,trimmed_MAE,NAE,mean_signed_error,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,reference,38.35,102.23,31.04,0.53,-34.82,6.03,8.02,11.78,25.95,74.67,356.58
1,S,reference,32.82,96.64,25.63,0.49,-23.65,8.52,7.28,8.98,16.42,57.46,339.89
2,SG_v1.1,eligible,32.76,95.85,25.62,0.53,-20.97,9.31,8.80,9.98,16.71,54.87,331.71
3,GPV_R4_v1,eligible,30.54,93.20,23.45,0.49,-18.23,7.91,8.44,10.35,15.07,46.75,323.80
4,GPV_K_v1,eligible,31.37,87.02,25.16,0.60,-10.94,10.00,11.38,12.25,21.08,49.03,278.27
5,H1b_v1,eligible,29.99,90.60,23.07,0.52,-16.12,9.58,8.82,10.33,14.61,44.90,311.18
6,H2_v1,diagnostic,31.98,86.92,25.82,0.64,-9.79,10.60,12.38,12.98,22.29,49.62,274.24



FULL VALIDATION (includes dev100 development images)


,system,role,MAE,RMSE,trimmed_MAE,NAE,mean_signed_error,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,reference,41.64,123.76,32.35,0.53,-38.13,5.96,7.90,12.03,26.60,74.02,397.55
1,S,reference,35.87,118.16,26.83,0.48,-26.89,8.21,7.15,8.98,17.02,56.79,378.20
2,SG_v1.1,eligible,35.03,114.89,26.26,0.52,-23.51,9.03,8.64,9.88,17.17,54.16,356.12
3,GPV_R4_v1,eligible,33.60,115.72,24.56,0.48,-21.60,7.66,8.33,10.27,15.21,46.16,364.38
4,GPV_K_v1,eligible,33.14,93.49,26.31,0.61,-12.77,10.56,11.07,12.03,21.76,48.32,293.71
5,H1b_v1,eligible,32.37,110.54,23.79,0.51,-18.88,9.17,8.75,10.27,14.65,44.28,340.03
6,H2_v1,diagnostic,33.13,91.44,26.51,0.65,-11.06,11.19,12.07,12.68,22.91,48.84,278.38



Held-out MAE minus dev100 MAE (negative = lower error on held-out):


,held-out − dev100
system,
B0,-42.34
S,-39.18
SG_v1.1,-29.14
GPV_R4_v1,-39.37
GPV_K_v1,-22.82
H1b_v1,-30.58
H2_v1,-14.81


In [11]:
# ============================================================
# Step C.6 — Paired bootstrap (10,000 resamples, seed 0)
# ============================================================

N_BOOT = 10_000

def boot_stats(df):
    gt = df["gt_count"].to_numpy(float)
    st = df["density_stratum"].to_numpy()
    n = len(df); k = max(1, int(round(0.01 * n)))
    rng = np.random.default_rng(0)
    idx = rng.integers(0, n, size=(N_BOOT, n))
    strat_idx = {}
    for j, s in enumerate(STRATA):
        pos = np.flatnonzero(st == s)
        r = np.random.default_rng(1000 + j)          # fixed per-stratum seeds
        strat_idx[s] = pos[r.integers(0, len(pos), size=(N_BOOT, len(pos)))]
    B = {}
    for s in SYSTEMS_ALL:
        ae = np.abs(df[s].to_numpy(float) - gt)
        sample = ae[idx]
        B[s] = {"MAE": sample.mean(axis=1),
                "RMSE": np.sqrt((sample ** 2).mean(axis=1)),
                "trimmed_MAE": np.sort(sample, axis=1)[:, :n - k].mean(axis=1),
                **{f"MAE_{x[:2]}": ae[strat_idx[x]].mean(axis=1) for x in STRATA}}
    return B

def ci(B, a, b, metric):
    d = B[a][metric] - B[b][metric]
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

t0 = time.time()
BOOT_HO = boot_stats(ho)
BOOT_FULL = boot_stats(full)
print(f"Bootstrap done in {time.time() - t0:.0f} s")

pairs = []
for i, a in enumerate(ELIGIBLE + ["H2_v1"]):
    for b in (ELIGIBLE + ["H2_v1"])[i + 1:]:
        for metric in ["MAE", "RMSE", "trimmed_MAE"]:
            lo, hi = ci(BOOT_HO, a, b, metric)
            obs = (rep_ho.set_index("system").loc[a, metric] - rep_ho.set_index("system").loc[b, metric])
            pairs.append({"A": a, "B": b, "metric": metric, "A−B": obs, "CI_low": lo, "CI_high": hi,
                          "significant": not (lo <= 0 <= hi)})
pairs_df = pd.DataFrame(pairs)
print("Held-out paired differences (A − B; negative = A better):")
display(pairs_df[pairs_df["metric"] == "MAE"].round(2))

Bootstrap done in 4 s
Held-out paired differences (A − B; negative = A better):


,A,B,metric,A−B,CI_low,CI_high,significant
0,SG_v1.1,GPV_R4_v1,MAE,2.22,1.34,3.09,True
3,SG_v1.1,GPV_K_v1,MAE,1.39,-0.91,3.88,False
6,SG_v1.1,H1b_v1,MAE,2.77,1.89,3.70,True
9,SG_v1.1,H2_v1,MAE,0.78,-1.47,3.19,False
12,GPV_R4_v1,GPV_K_v1,MAE,-0.83,-2.95,1.52,False
15,GPV_R4_v1,H1b_v1,MAE,0.55,0.14,0.98,True
18,GPV_R4_v1,H2_v1,MAE,-1.44,-3.62,0.98,False
21,GPV_K_v1,H1b_v1,MAE,1.38,-0.89,3.48,False
24,GPV_K_v1,H2_v1,MAE,-0.61,-1.09,-0.15,True
27,H1b_v1,H2_v1,MAE,-1.99,-4.10,0.33,False


In [12]:
# ============================================================
# Step C.7 — Pre-declared selection procedure (held-out)
# ============================================================

M = rep_ho.set_index("system")
log = []

# Stage 1 — accuracy
leader = min(ELIGIBLE, key=lambda s: M.loc[s, "MAE"])
tied = [leader] + [c for c in ELIGIBLE if c != leader
                   and ci(BOOT_HO, c, leader, "MAE")[0] <= 0 <= ci(BOOT_HO, c, leader, "MAE")[1]]
log.append(f"Stage 1 leader (lowest MAE): {leader}")
log.append(f"Stage 1 tied set (MAE CI includes 0): {tied}")
best_rmse = min(tied, key=lambda s: M.loc[s, "RMSE"])
stage1 = [c for c in tied if c == best_rmse or ci(BOOT_HO, c, best_rmse, "RMSE")[0] <= 0]
log.append(f"Stage 1 after RMSE filter (best RMSE {best_rmse}): {stage1}")

# Stage 2a — trimmed MAE
best_trim = min(stage1, key=lambda s: M.loc[s, "trimmed_MAE"])
stage2a = [c for c in stage1 if c == best_trim or ci(BOOT_HO, c, best_trim, "trimmed_MAE")[0] <= 0]
log.append(f"Stage 2a after trimmed-MAE filter (best {best_trim}): {stage2a}")

# Stage 2b — stratum balance (fail = significantly worse than the stratum's best in the current set)
fails = {c: [] for c in stage2a}
for s in STRATA:
    met = f"MAE_{s[:2]}"
    best_s = min(stage2a, key=lambda c: M.loc[c, met])
    for c in stage2a:
        if c != best_s and ci(BOOT_HO, c, best_s, met)[0] > 0:
            fails[c].append(s[:2])
stage2b = [c for c in stage2a if len(fails[c]) < 3]
log.append("Stage 2b stratum failures: " + ", ".join(f"{c}: {fails[c] or 'none'}" for c in stage2a))
if not stage2b:   # tie-handling: keep the candidate(s) with fewest failures
    fewest = min(len(v) for v in fails.values())
    stage2b = [c for c in stage2a if len(fails[c]) == fewest]
    log.append("Stage 2b removed every candidate — kept those with the fewest failures")
log.append(f"Stage 2b remaining: {stage2b}")

# Stage 3 — simplicity
primary = next(c for c in SIMPLICITY if c in stage2b)
rest = [c for c in SIMPLICITY if c in stage2b and c != primary]
fallback = rest[0] if rest else min([c for c in ELIGIBLE if c != primary], key=lambda s: M.loc[s, "MAE"])
log.append(f"Stage 3 (simplicity order {SIMPLICITY}): PRIMARY = {primary}; fallback = {fallback}")

clear = [c for c in ELIGIBLE if all(ci(BOOT_HO, c, o, "MAE")[1] < 0 for o in ELIGIBLE if o != c)]
log.append(f"Clear winner (MAE better than every other candidate, CI excludes 0): {clear or 'none'}")

print("=" * 80)
print("B4 SELECTION (pre-declared rule, held-out)")
print("=" * 80)
for line in log:
    print(line)

print("\nEffect sizes vs PRIMARY (held-out MAE, A − primary, 95% CI):")
for c in ELIGIBLE + ["H2_v1"]:
    if c != primary:
        lo, hi = ci(BOOT_HO, c, primary, "MAE")
        print(f"  {c:<10}: {M.loc[c, 'MAE'] - M.loc[primary, 'MAE']:+.2f}  [{lo:+.2f}, {hi:+.2f}]")

full_leader = rep_full.set_index("system").loc[ELIGIBLE, "MAE"].idxmin()
print(f"\nFull-val MAE leader (for comparison): {full_leader}"
      f"{'  (same as held-out leader)' if full_leader == leader else '  (DIFFERS from held-out leader)'}")

B4 SELECTION (pre-declared rule, held-out)
Stage 1 leader (lowest MAE): H1b_v1
Stage 1 tied set (MAE CI includes 0): ['H1b_v1', 'GPV_K_v1']
Stage 1 after RMSE filter (best RMSE GPV_K_v1): ['H1b_v1', 'GPV_K_v1']
Stage 2a after trimmed-MAE filter (best H1b_v1): ['H1b_v1']
Stage 2b stratum failures: H1b_v1: none
Stage 2b remaining: ['H1b_v1']
Stage 3 (simplicity order ['GPV_R4_v1', 'GPV_K_v1', 'SG_v1.1', 'H1b_v1']): PRIMARY = H1b_v1; fallback = GPV_R4_v1
Clear winner (MAE better than every other candidate, CI excludes 0): none

Effect sizes vs PRIMARY (held-out MAE, A − primary, 95% CI):
  SG_v1.1   : +2.77  [+1.89, +3.70]
  GPV_R4_v1 : +0.55  [+0.14, +0.98]
  GPV_K_v1  : +1.38  [-0.89, +3.48]
  H2_v1     : +1.99  [-0.33, +4.10]

Full-val MAE leader (for comparison): H1b_v1  (same as held-out leader)


In [13]:
# ============================================================
# Step C.8 — Save
# ============================================================

ho.to_csv(C_DIR / "B4_C_heldout_per_image.csv", index=False)
full.to_csv(C_DIR / "B4_C_fullval_per_image.csv", index=False)
rep_ho.to_csv(C_DIR / "B4_C_metrics_heldout.csv", index=False)
rep_full.to_csv(C_DIR / "B4_C_metrics_fullval.csv", index=False)
pairs_df.to_csv(C_DIR / "B4_C_bootstrap_pairs_heldout.csv", index=False)
with open(C_DIR / "B4_C_selection.json", "w") as f:
    json.dump({"date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "decision_set": "held-out (1,186)",
               "selection_log": log, "primary": primary, "fallback": fallback,
               "clear_winner": clear, "full_val_leader": full_leader,
               "precheck_pass": bool(PRECHECK_PASS), "n_boot": N_BOOT, "seed": 0,
               "test_set_accessed": False}, f, indent=2)
print("Saved to:", C_DIR)
print(f"\nPRIMARY ARCHITECTURE FOR B5: {primary}   (fallback: {fallback})")

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B4_HeldOut_FSC147_Validation/partC_evaluation

PRIMARY ARCHITECTURE FOR B5: H1b_v1   (fallback: GPV_R4_v1)


# B4 conclusion

## Pre-run checks — all PASS
- `dense_recovery` prompt port exact on frozen dev100 inputs (5,744/5,744 prompts within 1e-3 px; 100/100 images).
- Environment drift (CUDA 13.0): D6 max |diff| 7.2e-5; 100% of prompts within 1 px; GPV_R4 counts identical on 100/100 dev100 images.
- Held-out guided cache 1,186/1,186 (68,747 prompts).
- Strata reproduced 100/100; all five frozen functions reproduce their dev100 references 100/100.
- Sanity: B0 full-val MAE 41.64 = original B0 full-val result.

## Held-out results (1,186 images; decision set)

| | MAE | RMSE | Trimmed MAE | NAE | S1 | S4 | S5 | S6 |
|---|---|---|---|---|---|---|---|---|
| B0 | 38.35 | 102.2 | 31.04 | 0.53 | 6.03 | 25.95 | 74.67 | 356.6 |
| S | 32.82 | 96.6 | 25.63 | 0.49 | 8.52 | 16.42 | 57.46 | 339.9 |
| SG_v1.1 | 32.76 | 95.9 | 25.62 | 0.53 | 9.31 | 16.71 | 54.87 | 331.7 |
| GPV_R4_v1 | 30.54 | 93.2 | 23.45 | 0.49 | 7.91 | 15.07 | 46.75 | 323.8 |
| GPV_K_v1 | 31.37 | 87.0 | 25.16 | 0.60 | 10.00 | 21.08 | 49.03 | 278.3 |
| **H1b_v1** | **29.99** | 90.6 | **23.07** | 0.52 | 9.58 | **14.61** | **44.90** | 311.2 |
| *H2_v1 (diagnostic)* | 31.98 | 86.9 | 25.82 | 0.64 | 10.60 | 22.29 | 49.62 | 274.2 |

Full validation (1,286, includes dev100): H1b_v1 32.37, GPV_K_v1 33.14, GPV_R4_v1 33.60, SG_v1.1 35.03, B0 41.64 — **same leader as held-out**.

## Decision (pre-declared rule)
- **Stage 1:** leader H1b_v1; GPV_K_v1 tied on MAE (CI includes 0); RMSE filter removes neither.
- **Stage 2a:** GPV_K_v1 significantly worse on trimmed MAE → dropped.
- **Stage 2b:** H1b_v1 fails no strata.
- **Stage 3:** only H1b_v1 remains.

**PRIMARY: H1b_v1. Fallback: GPV_R4_v1.** No clear winner over all candidates: H1b beats GPV_R4_v1 by 0.55 MAE (95% CI 0.14–0.98) and SG_v1.1 by 2.77 (1.89–3.70); it is tied with GPV_K_v1 (1.38, −0.89–3.48).

## Findings
1. **SG's geometry gain was largely a dev100 effect.** On held-out, SG_v1.1 ≈ S without geometry (32.76 vs 32.82); G2 helps slightly in S5–S6 and costs slightly in S1–S3. The dev100 gain (72 → 62) came from a few dense images (e.g. 975.jpg).
2. **Guided proposal beats anchored correction.** GPV_R4_v1 beats SG_v1.1 by 2.22 MAE (CI 1.34–3.09), with lower NAE and better S1, S4, S5: steering SAM2 with semantics works better than patching an unguided count.
3. **Depth-edge multiplicity generalises in dense scenes, but so does its cost.** GPV_K_v1 has the best S6 (278 vs 324) and RMSE among eligible systems; Gate 1 false alarms cost more in the natural, sparse-dominated distribution (NAE 0.60).
4. **Routing does not generalise.** H2_v1 is significantly worse than GPV_K_v1 (CI −1.09 to −0.15): its dev100 advantage was 975.jpg. Architectures are complementary in principle (B3RK oracle) but cannot be routed with GT-free signals.
5. **H1b's advantage is real and balanced:** gains in S4–S6, small costs in S1–S2 — the same profile as on dev100.
6. **dev100 → held-out:** every system's MAE falls sharply (e.g. H1b −30.6) because dev100 deliberately oversamples dense scenes; the ranking pattern, not the level, transfers.
7. All systems still **undercount** (mean signed error −9.8 to −34.8), most in dense scenes.

## Context
Training-free, text-prompted: full-val MAE 32.37 vs 41.64 for unguided SAM2 (−22%). OmniCount reports 21.46 on FSC-147 **test** (text prompts); the like-for-like comparison is the FSC-147 test evaluation in B6.

## Next — B5 (multi-label adaptation, OmniCount-191 val, clean subset)
**H1b_v1 = GPV_R4_v1 + SG's local recovery and G2 inside merged masks**, so the fallback is nested inside the primary. Staged plan:
1. Adapt **GPV_R4** to multi-label (per-class prompts; class assignment of B0 masks).
2. Add **H1b's local stage** (class-adaptive A3c; recovery + G2 inside merged regions).

If stage 2 fails or time runs short, the adapted GPV_R4 (pre-declared fallback) is already working. **Backward-compatibility constraint:** with a single class, the adapted pipeline must reproduce the frozen B4 counts exactly (regression test on FSC held-out).